# Lecture 7: Protein Language Models - ESM Embeddings

This notebook demonstrates practical applications of ESM-2 (Evolutionary Scale Modeling):

1. Loading and using ESM-2 models
2. Extracting sequence embeddings
3. Zero-shot mutation effect prediction
4. Fine-tuning with LoRA for classification
5. Visualizing embeddings and attention maps

In [ ]:
# Install dependencies
# !pip install fair-esm torch transformers matplotlib scikit-learn seaborn

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Loading ESM-2 Models

ESM-2 comes in various sizes. We'll use the 650M parameter model for a good balance of performance and speed.

In [ ]:
# Method 1: Using fair-esm library
try:
    import esm
    
    # Load ESM-2 model (will download on first use)
    # Available models: esm2_t6_8M, esm2_t12_35M, esm2_t30_150M, esm2_t33_650M, esm2_t36_3B
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model = model.eval().to(device)
    
    print("ESM-2 model loaded successfully!")
    print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")
    USE_ESM = True
    
except ImportError:
    print("fair-esm not installed. Using mock functions for demonstration.")
    USE_ESM = False

In [ ]:
# Show alphabet (amino acid vocabulary)
if USE_ESM:
    print("ESM-2 Alphabet:")
    print(f"  Standard tokens: {alphabet.standard_toks}")
    print(f"  All tokens: {alphabet.all_toks}")
    print(f"  Vocabulary size: {len(alphabet)}")
    print(f"  Mask token: {alphabet.mask_idx} ('{alphabet.all_toks[alphabet.mask_idx]}')")
    print(f"  Padding token: {alphabet.padding_idx}")

## 2. Extracting Embeddings

ESM-2 produces embeddings at each position. We can aggregate these for sequence-level representations.

In [ ]:
# Example protein sequences
# Using well-known proteins for demonstration
sequences = [
    ("Insulin_A", "GIVEQCCTSICSLYQLENYCN"),
    ("Insulin_B", "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"),
    ("Ubiquitin", "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"),
    ("GFP_frag", "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRY"),
    ("Lysozyme_frag", "KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNL"),
]

print("Protein sequences:")
for name, seq in sequences:
    print(f"  {name}: {seq[:30]}... (length: {len(seq)})")

In [ ]:
def extract_embeddings(sequences, model, alphabet, batch_converter, layer=-1):
    """
    Extract ESM-2 embeddings for protein sequences.
    
    Args:
        sequences: List of (name, sequence) tuples
        model: ESM model
        alphabet: ESM alphabet
        batch_converter: Batch converter function
        layer: Which layer to extract (-1 for last)
    
    Returns:
        Dictionary with embeddings for each sequence
    """
    # Convert to batch format
    batch_labels, batch_strs, batch_tokens = batch_converter(sequences)
    batch_tokens = batch_tokens.to(device)
    
    # Get the layer index (ESM uses 1-indexed layers internally)
    num_layers = model.num_layers
    if layer < 0:
        layer = num_layers + layer + 1
    else:
        layer = layer
    
    # Forward pass
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[layer], return_contacts=True)
    
    # Extract representations
    token_embeddings = results["representations"][layer]  # [batch, seq_len+2, embed_dim]
    
    embeddings = {}
    for i, (name, seq) in enumerate(sequences):
        seq_len = len(seq)
        
        # Remove BOS and EOS tokens
        per_residue = token_embeddings[i, 1:seq_len+1, :].cpu()  # [L, D]
        
        embeddings[name] = {
            'per_residue': per_residue,
            'mean': per_residue.mean(dim=0),  # Mean pooling
            'cls': token_embeddings[i, 0, :].cpu(),  # CLS/BOS token
            'sequence': seq,
            'contacts': results["contacts"][i][:seq_len, :seq_len].cpu() if "contacts" in results else None
        }
    
    return embeddings

if USE_ESM:
    embeddings = extract_embeddings(sequences, model, alphabet, batch_converter)
    
    print("Extracted embeddings:")
    for name, emb in embeddings.items():
        print(f"  {name}:")
        print(f"    Per-residue shape: {emb['per_residue'].shape}")
        print(f"    Mean embedding shape: {emb['mean'].shape}")

### 2.1 Different Pooling Strategies

In [ ]:
def pool_embeddings(per_residue, strategy='mean'):
    """
    Pool per-residue embeddings to sequence-level.
    
    Args:
        per_residue: Per-residue embeddings [L, D]
        strategy: 'mean', 'max', 'first', 'last', or 'attention'
    
    Returns:
        Sequence embedding [D]
    """
    if strategy == 'mean':
        return per_residue.mean(dim=0)
    
    elif strategy == 'max':
        return per_residue.max(dim=0)[0]
    
    elif strategy == 'first':
        return per_residue[0]
    
    elif strategy == 'last':
        return per_residue[-1]
    
    elif strategy == 'attention':
        # Simple self-attention pooling
        # Compute attention weights based on embedding norms
        norms = per_residue.norm(dim=-1)  # [L]
        weights = torch.softmax(norms, dim=0).unsqueeze(-1)  # [L, 1]
        return (per_residue * weights).sum(dim=0)
    
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

if USE_ESM:
    # Compare pooling strategies for one protein
    test_emb = embeddings['Ubiquitin']['per_residue']
    
    print("Pooling comparison (first 5 dims):")
    for strategy in ['mean', 'max', 'first', 'last', 'attention']:
        pooled = pool_embeddings(test_emb, strategy)
        print(f"  {strategy:12s}: {pooled[:5].numpy().round(3)}")

## 3. Visualizing Embeddings

In [ ]:
if USE_ESM:
    # Collect mean embeddings
    mean_embeddings = []
    names = []
    for name, emb in embeddings.items():
        mean_embeddings.append(emb['mean'].numpy())
        names.append(name)
    
    mean_embeddings = np.array(mean_embeddings)
    
    # PCA for visualization
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(mean_embeddings)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=200, c=range(len(names)), cmap='viridis')
    
    for i, name in enumerate(names):
        plt.annotate(name, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                    xytext=(10, 5), textcoords='offset points', fontsize=12)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.title('ESM-2 Protein Embeddings (PCA)')
    plt.tight_layout()
    plt.show()

In [ ]:
if USE_ESM:
    # Visualize per-residue embeddings for one protein
    protein_name = "Ubiquitin"
    per_residue = embeddings[protein_name]['per_residue'].numpy()
    sequence = embeddings[protein_name]['sequence']
    
    # Reduce to 2D for visualization
    pca = PCA(n_components=2)
    residue_2d = pca.fit_transform(per_residue)
    
    # Color by position
    plt.figure(figsize=(12, 6))
    
    # Plot with position coloring
    scatter = plt.scatter(residue_2d[:, 0], residue_2d[:, 1], 
                         c=range(len(sequence)), cmap='viridis', s=100)
    
    # Add amino acid labels
    for i, aa in enumerate(sequence):
        plt.annotate(aa, (residue_2d[i, 0], residue_2d[i, 1]), 
                    fontsize=8, ha='center', va='center', color='white', fontweight='bold')
    
    plt.colorbar(scatter, label='Position in sequence')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(f'Per-Residue Embeddings: {protein_name}')
    plt.tight_layout()
    plt.show()

## 4. Zero-Shot Mutation Effect Prediction

ESM-2 can predict mutational effects without any training on fitness data.

In [ ]:
def predict_mutation_effect(sequence, position, wt_aa, mt_aa, model, alphabet, batch_converter):
    """
    Predict the effect of a mutation using masked language modeling.
    
    Score = log P(mutant | context) - log P(wildtype | context)
    
    Positive score = mutation is favorable
    Negative score = mutation is deleterious
    
    Args:
        sequence: Wild-type sequence
        position: 0-indexed mutation position
        wt_aa: Wild-type amino acid
        mt_aa: Mutant amino acid
        model, alphabet, batch_converter: ESM model components
    
    Returns:
        Log-likelihood ratio (mutation effect score)
    """
    # Verify position
    assert sequence[position] == wt_aa, f"Expected {wt_aa} at position {position}, got {sequence[position]}"
    
    # Create masked sequence
    masked_seq = list(sequence)
    masked_seq[position] = '<mask>'
    masked_seq = ''.join(masked_seq)
    
    # Tokenize
    _, _, tokens = batch_converter([('seq', masked_seq)])
    tokens = tokens.to(device)
    
    # Get predictions
    with torch.no_grad():
        output = model(tokens)
        logits = output['logits'][0, position + 1]  # +1 for BOS token
    
    # Get log probabilities
    log_probs = torch.log_softmax(logits, dim=-1)
    
    # Get token indices
    wt_idx = alphabet.get_idx(wt_aa)
    mt_idx = alphabet.get_idx(mt_aa)
    
    # Compute log-likelihood ratio
    llr = (log_probs[mt_idx] - log_probs[wt_idx]).item()
    
    return llr

if USE_ESM:
    # Example: Predict effect of mutations on Insulin A chain
    test_sequence = "GIVEQCCTSICSLYQLENYCN"
    position = 10  # Position to mutate (0-indexed)
    wt_aa = test_sequence[position]
    
    print(f"Sequence: {test_sequence}")
    print(f"Position: {position} (wild-type: {wt_aa})")
    print(f"\nMutation effects:")
    
    # Test all possible mutations
    standard_aas = 'ACDEFGHIKLMNPQRSTVWY'
    effects = {}
    
    for mt_aa in standard_aas:
        if mt_aa != wt_aa:
            score = predict_mutation_effect(
                test_sequence, position, wt_aa, mt_aa,
                model, alphabet, batch_converter
            )
            mutation = f"{wt_aa}{position+1}{mt_aa}"
            effects[mutation] = score
            
    # Sort by score
    sorted_effects = sorted(effects.items(), key=lambda x: x[1], reverse=True)
    
    print("\nTop 5 most favorable:")
    for mut, score in sorted_effects[:5]:
        print(f"  {mut}: {score:+.3f}")
    
    print("\nTop 5 most deleterious:")
    for mut, score in sorted_effects[-5:]:
        print(f"  {mut}: {score:+.3f}")

In [ ]:
def scan_all_positions(sequence, model, alphabet, batch_converter):
    """
    Compute mutation effects for all positions and amino acids.
    
    Returns:
        Matrix of shape [L, 20] with mutation scores
    """
    standard_aas = 'ACDEFGHIKLMNPQRSTVWY'
    L = len(sequence)
    
    scores = np.zeros((L, 20))
    
    for pos in range(L):
        wt_aa = sequence[pos]
        for j, mt_aa in enumerate(standard_aas):
            if mt_aa == wt_aa:
                scores[pos, j] = 0  # No change for wild-type
            else:
                scores[pos, j] = predict_mutation_effect(
                    sequence, pos, wt_aa, mt_aa,
                    model, alphabet, batch_converter
                )
    
    return scores, standard_aas

if USE_ESM:
    # Scan a short sequence
    short_seq = "GIVEQCCTSICSLYQLENYCN"  # Insulin A
    print(f"Scanning mutations for: {short_seq}")
    print("This may take a few minutes...")
    
    scores, aa_order = scan_all_positions(short_seq, model, alphabet, batch_converter)
    
    # Visualize as heatmap
    plt.figure(figsize=(20, 8))
    
    sns.heatmap(scores.T, cmap='RdBu_r', center=0,
                xticklabels=list(short_seq),
                yticklabels=list(aa_order))
    
    plt.xlabel('Position (Wild-type AA)')
    plt.ylabel('Mutant AA')
    plt.title('Mutation Effect Landscape (ESM-2 Zero-Shot)\nRed = Deleterious, Blue = Favorable')
    plt.tight_layout()
    plt.show()

## 5. Extracting Attention Maps

Attention maps from ESM can reveal structural contacts.

In [ ]:
def extract_attention(sequence, model, alphabet, batch_converter):
    """
    Extract attention maps from ESM model.
    
    Returns:
        Attention tensor [num_layers, num_heads, seq_len, seq_len]
    """
    _, _, tokens = batch_converter([('seq', sequence)])
    tokens = tokens.to(device)
    
    with torch.no_grad():
        results = model(tokens, repr_layers=[model.num_layers], need_head_weights=True)
    
    # Attention shape: [layers, batch, heads, seq_len, seq_len]
    attention = results['attentions']
    
    # Stack and remove batch dimension
    attention = torch.stack([a.squeeze(0) for a in attention], dim=0)
    
    # Remove BOS and EOS
    seq_len = len(sequence)
    attention = attention[:, :, 1:seq_len+1, 1:seq_len+1]
    
    return attention.cpu()

if USE_ESM:
    # Extract attention for a protein
    test_seq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRL"  # Part of Ubiquitin
    attention = extract_attention(test_seq, model, alphabet, batch_converter)
    
    print(f"Attention shape: {attention.shape}")
    print(f"  Layers: {attention.shape[0]}")
    print(f"  Heads: {attention.shape[1]}")
    print(f"  Sequence length: {attention.shape[2]}")

In [ ]:
if USE_ESM:
    # Visualize attention patterns
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Show attention from different layers and heads
    layers_to_show = [0, 10, 20, 32]  # Early to late layers
    
    for i, layer in enumerate(layers_to_show):
        # Mean over heads
        mean_attn = attention[layer].mean(0).numpy()
        
        axes[0, i].imshow(mean_attn, cmap='viridis')
        axes[0, i].set_title(f'Layer {layer+1} (mean over heads)')
        axes[0, i].set_xlabel('Key position')
        axes[0, i].set_ylabel('Query position')
        
        # Single head
        single_head = attention[layer, 0].numpy()  # First head
        axes[1, i].imshow(single_head, cmap='viridis')
        axes[1, i].set_title(f'Layer {layer+1}, Head 1')
        axes[1, i].set_xlabel('Key position')
        axes[1, i].set_ylabel('Query position')
    
    plt.suptitle('ESM-2 Attention Patterns', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
if USE_ESM:
    # Contact prediction from attention
    def attention_to_contacts(attention, symmetrize=True):
        """
        Convert attention maps to contact predictions.
        Uses average over layers and heads.
        """
        # Average over layers and heads
        avg_attn = attention.mean(dim=[0, 1]).numpy()
        
        if symmetrize:
            avg_attn = (avg_attn + avg_attn.T) / 2
        
        # Apply APC (Average Product Correction)
        row_mean = avg_attn.mean(axis=1, keepdims=True)
        col_mean = avg_attn.mean(axis=0, keepdims=True)
        overall_mean = avg_attn.mean()
        
        apc = (row_mean @ col_mean) / overall_mean
        corrected = avg_attn - apc
        
        return corrected
    
    contact_pred = attention_to_contacts(attention)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(contact_pred, cmap='Blues')
    plt.colorbar(label='Contact score')
    plt.xlabel('Residue')
    plt.ylabel('Residue')
    plt.title('Predicted Contacts from ESM-2 Attention')
    plt.tight_layout()
    plt.show()

## 6. Fine-Tuning with LoRA

LoRA (Low-Rank Adaptation) allows efficient fine-tuning by training only small adapter matrices.

In [ ]:
class LoRALayer(nn.Module):
    """
    Low-Rank Adaptation layer.
    
    For a pre-trained weight W, computes: W + BA
    where B and A are low-rank matrices.
    """
    
    def __init__(self, original_layer, r=8, alpha=16, dropout=0.1):
        """
        Args:
            original_layer: Pre-trained nn.Linear layer to adapt
            r: Rank of the adaptation
            alpha: Scaling factor
            dropout: Dropout probability
        """
        super().__init__()
        
        self.original_layer = original_layer
        self.r = r
        self.alpha = alpha
        
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        
        # Freeze original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False
        
        # LoRA matrices
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Scaling
        self.scaling = alpha / r
        
        # Initialize
        nn.init.kaiming_uniform_(self.lora_A, a=np.sqrt(5))
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x):
        # Original forward pass
        result = self.original_layer(x)
        
        # LoRA forward pass: x @ A^T @ B^T
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        
        return result + lora_out * self.scaling


# Example: Apply LoRA to a simple classifier
class ESMClassifierWithLoRA(nn.Module):
    """
    ESM-based classifier with LoRA adapters.
    """
    
    def __init__(self, esm_model, hidden_dim=1280, num_classes=2, lora_r=8):
        super().__init__()
        
        self.esm = esm_model
        
        # Freeze ESM parameters
        for param in self.esm.parameters():
            param.requires_grad = False
        
        # Classification head (trainable)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 4, num_classes)
        )
        
        # Note: In full implementation, you would also add LoRA to ESM attention layers
    
    def forward(self, tokens):
        # Get ESM embeddings
        with torch.no_grad():  # Keep ESM frozen
            results = self.esm(tokens, repr_layers=[self.esm.num_layers])
        
        embeddings = results['representations'][self.esm.num_layers]
        
        # Mean pooling (exclude special tokens)
        mask = (tokens != 0) & (tokens != 1) & (tokens != 2)  # Not pad/bos/eos
        mask = mask.unsqueeze(-1).float()
        pooled = (embeddings * mask).sum(1) / mask.sum(1)
        
        # Classify
        logits = self.classifier(pooled)
        
        return logits

print("LoRA layer defined!")
print("\nLoRA reduces trainable parameters by training only low-rank matrices.")
print(f"For a {1280}x{1280} weight matrix with rank {8}:")
print(f"  Original parameters: {1280*1280:,}")
print(f"  LoRA parameters: {8*1280 + 1280*8:,}")
print(f"  Reduction: {(8*1280 + 1280*8)/(1280*1280)*100:.2f}%")

In [ ]:
# Demonstration: Count parameters
def count_parameters(model):
    """Count total and trainable parameters"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

if USE_ESM:
    # Create classifier
    classifier = ESMClassifierWithLoRA(model, hidden_dim=1280, num_classes=2)
    
    total, trainable = count_parameters(classifier)
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Trainable ratio: {trainable/total*100:.4f}%")

## 7. Comparing Embedding Layers

In [ ]:
def extract_all_layers(sequence, model, alphabet, batch_converter):
    """
    Extract embeddings from all layers.
    """
    _, _, tokens = batch_converter([('seq', sequence)])
    tokens = tokens.to(device)
    
    # Get all layer representations
    all_layers = list(range(model.num_layers + 1))  # 0 to num_layers
    
    with torch.no_grad():
        results = model(tokens, repr_layers=all_layers)
    
    layer_embeddings = []
    seq_len = len(sequence)
    
    for layer in all_layers:
        emb = results['representations'][layer][0, 1:seq_len+1, :].cpu()  # [L, D]
        layer_embeddings.append(emb.mean(0))  # Mean pooling
    
    return torch.stack(layer_embeddings)  # [num_layers+1, D]

if USE_ESM:
    test_seq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRL"
    layer_embeddings = extract_all_layers(test_seq, model, alphabet, batch_converter)
    
    print(f"Embeddings from all layers: {layer_embeddings.shape}")

In [ ]:
if USE_ESM:
    # Compute similarity between consecutive layers
    layer_sims = []
    for i in range(len(layer_embeddings) - 1):
        sim = torch.cosine_similarity(
            layer_embeddings[i].unsqueeze(0),
            layer_embeddings[i+1].unsqueeze(0)
        ).item()
        layer_sims.append(sim)
    
    # Plot
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(layer_sims)+1), layer_sims, 'o-')
    plt.xlabel('Layer')
    plt.ylabel('Cosine Similarity')
    plt.title('Similarity Between Consecutive Layers')
    
    # Embedding norm by layer
    plt.subplot(1, 2, 2)
    norms = layer_embeddings.norm(dim=1).numpy()
    plt.plot(range(len(norms)), norms, 'o-')
    plt.xlabel('Layer')
    plt.ylabel('Embedding Norm')
    plt.title('Embedding Magnitude by Layer')
    
    plt.tight_layout()
    plt.show()

## 8. Summary

### Key Takeaways:

1. **ESM-2 produces rich embeddings** that capture evolutionary and structural information

2. **Different pooling strategies** (mean, max, CLS) suit different downstream tasks

3. **Zero-shot mutation prediction** uses log-likelihood ratios from masked language modeling

4. **Attention maps** can reveal structural contacts

5. **LoRA enables efficient fine-tuning** with <1% trainable parameters

6. **Layer choice matters**: Later layers capture more complex features

### Practical Recommendations:

- Use **esm2_t33_650M** for best balance of performance and speed
- Use **mean pooling** for general-purpose sequence embeddings
- Use **last layer** representations for most tasks
- Consider **middle layers** for structural tasks
- Apply **LoRA** when fine-tuning on limited data

In [ ]:
# Final summary
print("=" * 60)
print("ESM-2 NOTEBOOK SUMMARY")
print("=" * 60)

if USE_ESM:
    print(f"\nModel: ESM-2 650M")
    print(f"Embedding dimension: 1280")
    print(f"Number of layers: 33")
    print(f"Attention heads: 20")
    print(f"\nSequences analyzed: {len(sequences)}")
    print(f"\nKey capabilities demonstrated:")
    print("  - Embedding extraction")
    print("  - Zero-shot mutation prediction")
    print("  - Attention visualization")
    print("  - LoRA fine-tuning setup")
else:
    print("\nNote: Install fair-esm to run full examples")
    print("  pip install fair-esm")